# Paper 14: GPT-2: Language Models are Unsupervised Multitask Learners
**Authors**: Alec Radford, Jeffrey Wu, Rewon Child, David Luan, Dario Amodei, Ilya Sutskever (OpenAI, 2019)

---

## 1. The Historical Breaking Point: Fine-Tuning Bottlenecks & Generation Costs
Prior to GPT-2:
- Machine learning relied on supervised fine-tuning with task-specific heads for every downstream task.
- OpenAI asked: *Can a sufficiently large, autoregressively trained language model execute diverse tasks (translation, summarization, QA) purely from natural language prompts without parameter fine-tuning?*
- **Post-LN Instability**: Original Post-LN Transformers struggled to train past 12–24 layers without fragile learning rate warmups.
- **Quadratic Generation Complexity ($O(T^2)$)**: Generating sequences token-by-token recomputed keys and values across past context repeatedly.

## 2. Architectural Innovations
1. **Pre-Layer Normalization (Pre-LN)**:
   $$\mathbf{x} \leftarrow \mathbf{x} + \text{CausalSelfAttn}(\text{LayerNorm}(\mathbf{x}))$$
   $$\mathbf{x} \leftarrow \mathbf{x} + \text{MLP}(\text{LayerNorm}(\mathbf{x}))$$
   LayerNorm is moved to the input of each sub-block, with an additional LayerNorm after the final block.
2. **Residual Initialization Scaling**: Residual projections are scaled by $\frac{1}{\sqrt{2N}}$ where $N$ is the layer count.
3. **KV-Cache Inference**: Key and Value projections for past tokens are cached, allowing $O(1)$ computation per generated token!

In this notebook, you will test and verify:
- `GPT2Attention` with causal masking and KV-caching
- `GPT2MLP` with GELU activation
- `GPT2Block` and `GPT2Model`
- `GPT2LMHeadModel` with tied weights and `generate()`

---
## Setup & Autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from model import (
    GPT2Attention,
    GPT2MLP,
    GPT2Block,
    GPT2Model,
    GPT2LMHeadModel,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

---
## 1. Test Causal Masking Invariant
In an autoregressive decoder, modifying future tokens at step $t+1$ must **NEVER** affect earlier outputs at step $t$.

In [ ]:
torch.manual_seed(42)
d_model, num_heads = 64, 4
attn = GPT2Attention(d_model=d_model, num_heads=num_heads, max_positions=128, dropout=0.0)
attn.eval()

x1 = torch.randn(1, 6, d_model)
x2 = x1.clone()
x2[:, 3:, :] = torch.randn(1, 3, d_model)  # Alter tokens 3, 4, 5

with torch.no_grad():
    out1, _ = attn(x1)
    out2, _ = attn(x2)

diff = (out1[:, :3, :] - out2[:, :3, :]).abs().max().item()
print(f"Difference in tokens 0..2 when changing tokens 3..5: {diff:.2e}")
assert diff < 1e-5
print("Causal Masking Invariant PASSED!")

---
## 2. Test KV-Cache Numerical Equivalence
Verify that generating tokens step-by-step using cached keys and values produces **identical** logits to running the full sequence in a single forward pass.

In [ ]:
torch.manual_seed(42)
model = GPT2LMHeadModel(vocab_size=50, max_positions=64, d_model=64, num_layers=2, num_heads=4, dropout=0.0)
model.eval()

sequence = torch.randint(0, 50, (1, 8))

# Full forward
with torch.no_grad():
    logits_full, _ = model(sequence, use_cache=False)

# Cached forward step by step
past_key_values = None
cached_logits = []
with torch.no_grad():
    for t in range(sequence.size(1)):
        tok_logits, past_key_values = model(sequence[:, t:t+1], past_key_values=past_key_values, use_cache=True)
        cached_logits.append(tok_logits)

logits_cached = torch.cat(cached_logits, dim=1)
diff = (logits_full - logits_cached).abs().max().item()
print(f"Maximum difference between full and KV-cached logits: {diff:.2e}")
assert diff < 1e-4
print("KV-Cache Equivalence PASSED!")

---
## 3. Single-Batch Overfit: Autoregressive Next-Token Prediction
Train `GPT2LMHeadModel` on a small synthetic sequence to verify next-token cross-entropy convergence.

In [ ]:
torch.manual_seed(42)
vocab_size, d_model = 30, 64
B, T = 2, 8

model = GPT2LMHeadModel(vocab_size=vocab_size, max_positions=32, d_model=d_model, num_layers=2, num_heads=4, dropout=0.0)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

input_ids = torch.randint(0, vocab_size, (B, T))
targets = input_ids[:, 1:].contiguous()

losses = []
for epoch in range(80):
    optimizer.zero_grad()
    logits, _ = model(input_ids)
    shift_logits = logits[:, :-1, :].contiguous()
    loss = loss_fn(shift_logits.view(-1, vocab_size), targets.view(-1))
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f}")

plt.figure(figsize=(7, 3))
plt.plot(losses, marker='o', markersize=3, color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('Autoregressive Loss')
plt.title('GPT-2 Next-Token Memorization Curve')
plt.grid(True, alpha=0.3)
plt.show()

assert losses[-1] < 0.1
print(f"Overfit check PASSED! Final loss: {losses[-1]:.4f}")

---
## 4. Test Autoregressive Generation with KV-Cache

In [ ]:
prompt = torch.tensor([[1, 5, 14]])
generated = model.generate(prompt, max_new_tokens=10, temperature=0.8, top_k=5, use_cache=True)
print("Input prompt:     ", prompt[0].tolist())
print("Generated sequence:", generated[0].tolist())
assert generated.shape == (1, 13)
print("Autoregressive generation PASSED!")